In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error, r2_score

data = pd.read_csv("/content/discharge.csv")
print(data.head())

   Voltage_measured  Current_measured  Temperature_measured  Current_charge  \
0          3.974871         -2.012528             24.389085          1.9982   
1          3.951717         -2.013979             24.544752          1.9982   
2          3.934352         -2.011144             24.731385          1.9982   
3          3.920058         -2.013007             24.909816          1.9982   
4          3.907904         -2.014400             25.105884          1.9982   

   Voltage_charge     Time  Capacity  id_cycle       type  \
0           3.062   35.703  1.856487         1  discharge   
1           3.030   53.781  1.856487         1  discharge   
2           3.011   71.922  1.856487         1  discharge   
3           2.991   90.094  1.856487         1  discharge   
4           2.977  108.281  1.856487         1  discharge   

   ambient_temperature    time Battery  
0                   24  2008.0   B0005  
1                   24  2008.0   B0005  
2                   24  2008.0   B0

In [3]:
data.info()
data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 169766 entries, 0 to 169765
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Voltage_measured      169766 non-null  float64
 1   Current_measured      169766 non-null  float64
 2   Temperature_measured  169766 non-null  float64
 3   Current_charge        169766 non-null  float64
 4   Voltage_charge        169766 non-null  float64
 5   Time                  169766 non-null  float64
 6   Capacity              169766 non-null  float64
 7   id_cycle              169766 non-null  int64  
 8   type                  169766 non-null  object 
 9   ambient_temperature   169766 non-null  int64  
 10  time                  169766 non-null  float64
 11  Battery               169766 non-null  object 
dtypes: float64(8), int64(2), object(2)
memory usage: 15.5+ MB


,Voltage_measured,Current_measured,Temperature_measured,Current_charge,Voltage_charge,Time,Capacity,id_cycle,ambient_temperature,time
count,169766.000000,169766.000000,169766.000000,169766.000000,169766.000000,169766.000000,169766.000000,169766.000000,169766.0,169766.0
mean,3.503756,-2.004652,31.985477,1.998999,2.573131,1446.758949,1.584585,80.995187,24.0,2008.0
std,0.245871,0.009801,3.780617,0.000704,0.238604,850.462795,0.189489,45.504296,0.0,0.0
min,1.737030,-2.029098,22.372620,1.998000,0.890000,19.468000,1.153818,1.000000,24.0,2008.0
25%,3.389014,-2.011621,29.419671,1.998200,2.458000,722.016000,1.431808,44.000000,24.0,2008.0
50%,3.507333,-2.009471,31.931205,1.999000,2.578000,1424.813000,1.570257,78.000000,24.0,2008.0
75%,3.665754,-1.991235,34.756413,2.000000,2.733000,2131.113250,1.750291,118.000000,24.0,2008.0
max,4.035025,-1.974808,42.083729,2.000000,3.097000,3690.234000,2.035338,168.000000,24.0,2008.0


In [4]:
data = data.dropna()

In [10]:
features = ['id_cycle','Voltage_measured','Current_measured','Temperature_measured']
target = 'Capacity'

X = data[features]
y = data[target]

In [11]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [13]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [14]:
dt = DecisionTreeRegressor()
dt.fit(X_train, y_train)

DecisionTreeRegressor()

In [15]:
rf = RandomForestRegressor(n_estimators=200)
rf.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200)

In [16]:
svm = SVR()
svm.fit(X_train, y_train)

SVR()

In [17]:
models = {
    "Linear Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "SVM": svm
}

for name, model in models.items():

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    print(name)
    print("MAE:", mae)
    print("R2 Score:", r2)
    print("------------------")

Linear Regression
MAE: 0.04410974243963468
R2 Score: 0.9126887060660388
------------------
Decision Tree
MAE: 0.012825199952052835
R2 Score: 0.9588884330980233
------------------
Random Forest
MAE: 0.013100779558324262
R2 Score: 0.9785754995403492
------------------
SVM
MAE: 0.03650598080560012
R2 Score: 0.9420669187188531
------------------


In [18]:
initial_capacity = 2.0

def calculate_soh(capacity):

    soh = (capacity / initial_capacity) * 100

    return soh

In [19]:
def estimate_rul(soh):

    rul = (soh/100) * 5

    return rul

In [20]:
def recommend_use(soh):

    if soh >= 80:
        return "Electric Vehicle Reuse"

    elif soh >= 70:
        return "Solar Energy Storage"

    elif soh >= 60:
        return "Backup Power Systems"

    else:
        return "Recycle Battery"

In [21]:
def pack_group(soh):

    if soh >= 80:
        return "Group with high performance pack"

    elif soh >= 70:
        return "Group with solar storage pack"

    elif soh >= 60:
        return "Group with backup pack"

    else:
        return "Do not group"

In [24]:
def battery_system():

    cycle = int(input("Enter Charge Cycles (0 - 3000): "))
    voltage = float(input("Enter Battery Voltage (2.5 - 4.2 V): "))
    current = float(input("Enter Current (-100 to 100 A): "))
    temperature = float(input("Enter Temperature (0 - 60 °C): "))
    user_data = np.array([[cycle, voltage, current, temperature]])

    user_scaled = scaler.transform(user_data)

    capacity_pred = rf.predict(user_scaled)[0]

    soh = calculate_soh(capacity_pred)

    rul = estimate_rul(soh)

    reuse = recommend_use(soh)

    group = pack_group(soh)

    print("\n------ BATTERY AI RESULT ------")

    print("Predicted Capacity:", round(capacity_pred,2))

    print("Battery Health (SOH):", round(soh,2),"%")

    print("Remaining Useful Life:", round(rul,2),"years")

    print("Recommended Application:", reuse)

    print("Battery Pack Group:", group)

    print("IoT Monitoring: Required for safety")

In [25]:
battery_system()

Enter Charge Cycles (0 - 3000): 2000
Enter Battery Voltage (2.5 - 4.2 V): 3.8
Enter Current (-100 to 100 A): 58
Enter Temperature (0 - 60 °C): 40

------ BATTERY AI RESULT ------
Predicted Capacity: 1.43
Battery Health (SOH): 71.62 %
Remaining Useful Life: 3.58 years
Recommended Application: Solar Energy Storage
Battery Pack Group: Group with solar storage pack
IoT Monitoring: Required for safety


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
